# Analisis Crawl Web dan PageRank Antar Halaman

**Tugas Web Structure Mining - Notebook 3**

Notebook ini mendemonstrasikan konsep web crawling untuk membangun directed graph antar halaman web, lalu menghitung PageRank untuk menemukan halaman/domain yang paling berpengaruh.

## Daftar Isi
1. Konsep Web Crawling
2. Persiapan Library
3. Memanggil Modul Crawler
4. Menjalankan Crawler pada URL Demo
5. Membangun Directed Graph
6. Menghitung PageRank
7. Visualisasi Graph
8. Top 10 PageRank
9. Ekspor Hasil ke File
10. Penjelasan Algoritma PageRank
11. Kesimpulan dan Referensi

## Referensi PPT Dosen
Materi ini mengacu pada PPT **Web Structure Mining** bagian:
- *2.2 Prestise* - Rank Prestise dan PageRank (rumus dari Bing Liu, Chapter 7.3)
- *1. Mendeskripsikan Graf* - konsep dasar graph, adjacency matrix, derajat

## 1. Konsep Web Crawling

Web crawling adalah proses otomatis untuk menelusuri halaman web, mengikuti tautan (link) dari satu halaman ke halaman lain, dan mengumpulkan informasi.

### Alur Kerja Crawler

1. **Seed URL**: Mulai dari satu URL awal (mis. https://trunojoyo.ac.id)
2. **Fetch**: Download HTML halaman tersebut
3. **Parse**: Ekstrak semua tag `<a href="...">` dari HTML
4. **Filter**: Hanya ambil link yang valid dan sesuai aturan (mis. same-domain)
5. **Queue**: Tambahkan link baru ke antrian untuk di-crawl
6. **Repeat**: Ulangi sampai depth maksimum atau jumlah halaman tercapai

### Konsep Depth

- **Depth 0**: hanya halaman seed
- **Depth 1**: seed + semua halaman yang di-link dari seed
- **Depth 2**: halaman depth 1 + semua halaman yang di-link dari halaman depth 1

Makin dalam depth, makin banyak halaman ditemukan, tapi waktu juga makin lama.

### Setelah Crawl Selesai

Hasil crawler adalah kumpulan halaman + tautan antar halaman. Kumpulan ini bisa dibentuk menjadi **directed graph** dengan halaman sebagai node dan tautan sebagai edge. Pada graph ini, kita bisa menghitung **PageRank** untuk menemukan halaman paling berpengaruh.

## 2. Persiapan Library

Library yang dibutuhkan:
- `networkx` untuk analisis graph
- `requests` untuk fetch HTML
- `beautifulsoup4` untuk parse HTML
- `matplotlib` untuk visualisasi
- `pandas` untuk tabel rapi

Modul internal (di folder yang sama):
- `crawler.py` berisi class `Crawler` untuk crawling
- `graph_builder.py` berisi fungsi untuk bangun graph dan hitung PageRank

In [ ]:
import sys
import os

# Tambahkan folder web-crawler-pagerank ke path
# (asumsi notebook ini ada di root folder, dan web-crawler-pagerank ada di subfolder)
PROJECT_DIR = 'web-crawler-pagerank'
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

import networkx as nx
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

from crawler import Crawler, normalize_url, get_domain
from graph_builder import build_graph, pagerank, pagerank_manual, compute_metrics, graph_summary, top_n_pagerank, export_graph

print('NetworkX version :', nx.__version__)
print('Pandas version   :', pd.__version__)
print('Modul crawler    : OK')
print('Modul graph_builder: OK')

## 3. Menjalankan Crawler

Kita akan menjalankan crawler pada URL demo. Untuk notebook ini, gunakan **depth kecil** agar cepat dan tidak membebani server target.

### Parameter Crawler

| Parameter | Default | Keterangan |
|-----------|---------|------------|
| `max_depth` | 2 | Kedalaman crawl |
| `max_pages` | 50 | Total halaman maksimum |
| `per_domain` | 5 | Batas halaman per domain |
| `delay` | 0.5 | Jeda antar request (detik) |
| `respect_robots` | True | Patuhi robots.txt |
| `restrict_to_seed_domain` | False | Hanya crawl domain yang sama dengan seed |

In [ ]:
# Konfigurasi crawler untuk demo kecil
# Ganti seed_url dengan website yang ingin dianalisis
seed_url = 'https://example.com'  # contoh aman untuk demo

crawler = Crawler(
    max_depth=1,        # depth kecil biar cepat
    max_pages=15,       # batas 15 halaman
    per_domain=5,       # max 5 halaman per domain
    delay=0.5,         # jeda 0.5 detik antar request
    workers=2,         # 2 thread paralel
    respect_robots=True,
    max_seconds=30,    # timeout 30 detik
    restrict_to_seed_domain=False,
)

# Jalankan crawler
import time
t0 = time.time()
crawler.crawl([seed_url])
elapsed = time.time() - t0

print(f'Crawl selesai dalam {elapsed:.2f} detik')
print(f'Halaman di-crawl : {len(crawler.pages)}')
print(f'Domain ditemukan: {len(crawler.pages_crawled)}')
print(f'Link diekstrak  : {len(crawler.page_links)}')
print(f'Edge antar-domain: {len(crawler.edges)}')

In [ ]:
# Lihat ringkasan crawler
summary = crawler.summary()
for k, v in summary.items():
    print(f'  {k}: {v}')

## 4. Membangun Directed Graph

Dari hasil crawler, kita bangun directed graph antar domain. Setiap node adalah domain, dan edge mengindikasikan ada tautan dari satu domain ke domain lain.

In [ ]:
# Bangun directed graph dari hasil crawler
G = build_graph(crawler, weighted=True)

print(f'Jumlah node (domain): {G.number_of_nodes()}')
print(f'Jumlah edge         : {G.number_of_edges()}')
print()
print('Daftar node:')
for node in sorted(G.nodes()):
    print(f'  - {node}')
print()
print('Daftar edge (source -> target, weight):')
for u, v, d in G.edges(data=True):
    print(f'  {u} -> {v} (weight: {d.get("weight", 1)})')

In [ ]:
# Matriks ketetanggaan (adjacency matrix)
nodes_sorted = sorted(G.nodes())
A = nx.adjacency_matrix(G, nodelist=nodes_sorted).todense()
df_A = pd.DataFrame(A, index=nodes_sorted, columns=nodes_sorted, dtype=int)

print('Adjacency Matrix (matriks ketetanggaan antar domain):')
print('=' * 60)
print(df_A)
print('=' * 60)
print()
print('Catatan: A_ij = jumlah tautan dari domain i ke domain j')

## 5. Menghitung PageRank

Rumus PageRank (Bing Liu, Chapter 7.3):

$$PR(i) = \frac{(1 - d)}{N} + d \sum_{(j,i) \in E} \frac{PR(j)}{D_{out}(j)}$$

Kita akan menghitung dengan dua cara:
1. `nx.pagerank()` - implementasi NetworkX (production)
2. `pagerank_manual()` - implementasi iterasi manual (edukasi)

In [ ]:
# Hitung PageRank dengan NetworkX
pr_nx = pagerank(G, damping=0.85)

# Hitung PageRank dengan iterasi manual
pr_manual = pagerank_manual(G, damping=0.85, max_iter=100)

# Bandingkan keduanya
comparison = pd.DataFrame({
    'Domain': list(pr_nx.keys()),
    'PR (NetworkX)': [pr_nx[d] for d in pr_nx.keys()],
    'PR (Manual)': [pr_manual[d] for d in pr_nx.keys()],
})
comparison = comparison.sort_values('PR (NetworkX)', ascending=False).reset_index(drop=True)
comparison['Selisih'] = abs(comparison['PR (NetworkX)'] - comparison['PR (Manual)'])

print('Perbandingan PageRank (NetworkX vs Manual Iteration):')
print('=' * 70)
print(comparison.to_string(index=False))
print('=' * 70)
print()
print(f'Total PR NetworkX: {sum(pr_nx.values()):.6f}')
print(f'Total PR Manual   : {sum(pr_manual.values()):.6f}')

In [ ]:
# Hitung metrik lengkap untuk setiap node
metrics_df = compute_metrics(G, pr_nx)
print('Metrik lengkap tiap domain (diurutkan berdasarkan PageRank):')
print('=' * 70)
print(metrics_df.to_string(index=False))

## 6. Visualisasi Graph

Ukuran node proporsional terhadap PageRank. Node kuning = seed domain.

In [ ]:
# Visualisasi dengan matplotlib
plt.figure(figsize=(12, 8))
pos = nx.spring_layout(G, seed=42, k=2.0)

# Ukuran node berdasarkan PageRank
node_sizes = [300 + 3000 * pr_nx.get(node, 0) for node in G.nodes()]

# Warna node: kuning untuk seed, biru untuk lainnya
seed_domain = get_domain(normalize_url(seed_url))
node_colors = ['#fbbf24' if n == seed_domain else '#6cb4ee' for n in G.nodes()]

nx.draw_networkx_edges(G, pos, edge_color='gray', arrows=True, arrowsize=20, alpha=0.6)
nx.draw_networkx_nodes(G, pos, node_size=node_sizes, node_color=node_colors, edgecolors='black', linewidths=1.5)
nx.draw_networkx_labels(G, pos, font_size=10, font_weight='bold')

# Tambah edge label (weight)
edge_labels = {(u, v): d.get('weight', 1) for u, v, d in G.edges(data=True)}
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=8)

plt.title(f'Directed Graph Antar Domain\nNode kuning = seed ({seed_domain})\nUkuran node = proporsional terhadap PageRank',
          fontsize=12, fontweight='bold')
plt.axis('off')
plt.tight_layout()
plt.savefig('images/crawl_graph.png', dpi=150, bbox_inches='tight')
plt.show()
print('Graph disimpan ke images/crawl_graph.png')

## 7. Top 10 PageRank

Tampilkan 10 domain dengan PageRank tertinggi dalam format tabel dan bar chart.

In [ ]:
# Top 10 PageRank
top10_df = top_n_pagerank(pr_nx, n=10)
print('Top 10 Domain dengan PageRank Tertinggi:')
print('=' * 50)
print(top10_df.to_string(index=False))
print('=' * 50)
print(f'\nDomain paling berpengaruh: {top10_df.iloc[0]["node"]} (PR = {top10_df.iloc[0]["page_rank"]:.6f})')

In [ ]:
# Bar chart Top 10 PageRank
plt.figure(figsize=(10, 6))
plt.barh(top10_df['node'][::-1], top10_df['page_rank'][::-1], color='#6cb4ee', edgecolor='black')
plt.xlabel('PageRank', fontsize=12)
plt.ylabel('Domain', fontsize=12)
plt.title('Top 10 Domain dengan PageRank Tertinggi', fontsize=14, fontweight='bold')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('images/crawl_pagerank_top10.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Ringkasan Metrik Graph

Metrik level graph membantu memahami kompleksitas jaringan yang terbentuk.

In [ ]:
# Ringkasan graph
graph_stats = graph_summary(G)
print('Ringkasan Graph:')
print('=' * 50)
for k, v in graph_stats.items():
    print(f'  {k:25} : {v}')
print('=' * 50)

## 9. Ekspor Hasil ke File

Hasil analisis bisa diekspor ke beberapa format:
- **GraphML** - untuk dibuka di Gephi/Cytoscape
- **JSON** - untuk integrasi web
- **CSV** - untuk analisis di Excel/pandas

In [ ]:
# Export hasil ke file
export_paths = export_graph(G, pr_nx, 'web-crawler-pagerank/exports', prefix='notebook_crawl')

print('Hasil ekspor:')
print('=' * 60)
for fmt, path in export_paths.items():
    size = os.path.getsize(path)
    print(f'  {fmt:12s} : {path} ({size} bytes)')
print('=' * 60)

## 10. Penjelasan Algoritma PageRank

### Intuisi

PageRank mengukur **pentingnya** sebuah halaman web berdasarkan struktur tautan. Intuisinya:

- Setiap halaman memiliki 'suara' yang besarnya proporsional dengan PageRank-nya sendiri.
- Suara ini **dibagi rata** ke semua halaman yang ditautnya.
- Sebuah halaman menerima suara dari semua halaman yang menautnya.
- Halaman yang ditauni banyak halaman penting akan menjadi penting juga.

Ini adalah **circular reasoning**, jadi diselesaikan dengan **iterasi**:
1. Inisialisasi semua PageRank = 1/N (seragam)
2. Update setiap PageRank dengan rumus
3. Ulangi sampai konvergen (perubahan sangat kecil)

### Rumus

$$PR(i) = \frac{(1 - d)}{N} + d \sum_{(j,i) \in E} \frac{PR(j)}{D_{out}(j)}$$

Dengan:
- $d$ = damping factor (default 0.85)
- $N$ = jumlah node
- $D_{out}(j)$ = out-degree dari node $j$
- $(1-d)/N$ = teleportation probability (user lompat ke halaman acak)

### Penanganan Dangling Node

Dangling node = node dengan out-degree = 0. PR-nya didistribusikan ke semua node secara merata.

### Damping Factor

- $d = 0.85$ berarti 85% user mengikuti link, 15% lompat ke halaman acak
- Nilai ini dipilih Google karena memberi konvergensi cepat dan hasil stabil

## 11. Kesimpulan

Dalam notebook ini kita telah:

1. Menjalankan web crawler pada URL demo dengan parameter terbatas
2. Mengekstrak semua tautan dari setiap halaman yang di-crawl
3. Membangun directed graph antar domain dari hasil crawler
4. Menghitung PageRank dengan dua metode (NetworkX dan iterasi manual) dan membandingkan hasilnya
5. Menampilkan top 10 domain berdasarkan PageRank
6. Menghitung metrik graph (density, connectivity, diameter)
7. Mengekspor hasil ke format GraphML, JSON, dan CSV

## 12. Referensi

1. PPT Dosen, **Web Structure Mining**. Bab Mendeskripsikan Graf dan Prestise.
2. Bing Liu, **Web Data Mining**, Chapter 7.3 (PageRank).
3. Page, L., Brin, S., Motwani, R., & Winograd, T. (1999). *The PageRank Citation Ranking: Bringing Order to the Web*. Stanford InfoLab.
4. NetworkX Documentation: https://networkx.org/documentation/stable/
5. BeautifulSoup Documentation: https://www.crummy.com/software/BeautifulSoup/bs4/doc/

## 13. Catatan Penggunaan

- Notebook ini mendemonstrasikan konsep. Untuk analisis real, gunakan **Web Crawler PageRank Analyzer** (file `web-crawler-pagerank/app.py`) yang merupakan Streamlit app dengan UI interaktif.
- Jalankan dengan: `streamlit run web-crawler-pagerank/app.py`
- Pastikan semua library terinstall: `pip install -r web-crawler-pagerank/requirements.txt`